<a href="https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule: Create a simple baseline action score using signals available at decision time. Higher scores indicate higher-priority content opportunities.

Reason codes:

STALE_CONTENT — the content shows a strong staleness signal.
HIGH_OPPORTUNITY — the content shows a strong opportunity signal.
LOW_PRIORITY — the available signals do not indicate an immediate action.

Actions:

REFRESH — update or improve the content.
REVIEW — manually inspect the content before acting.
NO_ACTION — no immediate action recommended.


In [ ]:
import pandas as pd
from pathlib import Path
import os

# ============================================================
# ML-07 — Baseline Action Score
# Section 2: Build the ranked queue
# ============================================================

# ------------------------------------------------------------
# 1. Locate the starter dataset automatically
# ------------------------------------------------------------

possible_paths = [
    "/content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
]

data_path = None

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

# If the dataset is not in the common locations, search /content
if data_path is None:
    for root, dirs, files in os.walk("/content"):
        if "content_refresh_anonymized.csv" in files:
            data_path = os.path.join(root, "content_refresh_anonymized.csv")
            break

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Make sure your FlyRank repository/data is available in Colab."
    )

print("Dataset found at:")
print(data_path)

# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(data_path)

print(f"\nRows loaded: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ------------------------------------------------------------
# 3. Convert required signals to numeric
# ------------------------------------------------------------

numeric_cols = [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Missing numeric values become 0 for this simple baseline
for col in numeric_cols:
    df[col] = df[col].fillna(0)

# ------------------------------------------------------------
# 4. Create signal scores
# ------------------------------------------------------------

# Staleness:
# Higher days_since_last_update = more stale
df["staleness_score"] = df["days_since_last_update"].rank(pct=True)

# Search volume:
# Higher search volume = larger potential opportunity
df["volume_score"] = df["search_volume"].rank(pct=True)

# Recent impressions:
# Higher recent impressions = stronger observed demand
df["impression_score"] = df["impressions_last_30d"].rank(pct=True)

# Ranking opportunity:
# Pages ranking roughly positions 4–20 are potentially actionable
df["position_score"] = (
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 20)
).astype(float)

# ------------------------------------------------------------
# 5. ONE simple baseline score
# ------------------------------------------------------------

df["baseline_score"] = (
    0.40 * df["staleness_score"]
    + 0.30 * df["volume_score"]
    + 0.20 * df["impression_score"]
    + 0.10 * df["position_score"]
)

# ------------------------------------------------------------
# 6. Assign ONE reason code
# ------------------------------------------------------------

df["reason_code"] = "LOW_PRIORITY"

# Strong staleness signal
df.loc[
    df["staleness_score"] >= 0.75,
    "reason_code"
] = "STALE_CONTENT"

# Strong opportunity signal
df.loc[
    (df["staleness_score"] < 0.75)
    & (df["volume_score"] >= 0.75)
    & (df["position_score"] == 1),
    "reason_code"
] = "HIGH_OPPORTUNITY"

# ------------------------------------------------------------
# 7. Assign action label
# ------------------------------------------------------------

df["action"] = "NO_ACTION"

df.loc[
    df["reason_code"] == "STALE_CONTENT",
    "action"
] = "REFRESH"

df.loc[
    df["reason_code"] == "HIGH_OPPORTUNITY",
    "action"
] = "REVIEW"

# ------------------------------------------------------------
# 8. Rank the complete queue
# ------------------------------------------------------------

df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# ------------------------------------------------------------
# 9. Create final ranked queue
# ------------------------------------------------------------

output_cols = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

queue = df[output_cols].copy()

# ------------------------------------------------------------
# 10. Write required CSV
# ------------------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

# ------------------------------------------------------------
# 11. Show results
# ------------------------------------------------------------

print(f"\nRows ranked: {len(queue):,}")
print(f"Output written to: {output_path}")

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 20 ranked rows:")
display(queue.head(20))

Dataset found at:
/content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv

Rows loaded: 30,000
Columns: 44

Rows ranked: 30,000
Output written to: work/outputs/baseline_action_score.csv

Reason-code counts:
reason_code
LOW_PRIORITY        17913
STALE_CONTENT        9091
HIGH_OPPORTUNITY     2996
Name: count, dtype: int64

Top 20 ranked rows:


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume,impressions_last_30d,avg_position
0,1,content_5fe46e04994d,client_4e07408562,0.933433,STALE_CONTENT,REFRESH,104,1900.0,120791,4.2
1,2,content_e6955a2c59dc,client_4e07408562,0.930112,STALE_CONTENT,REFRESH,104,3600.0,11101,7.1
2,3,content_b242bb46cb5e,client_3fdba35f04,0.929402,STALE_CONTENT,REFRESH,104,1600.0,15159,5.5
3,4,content_2e0b3dc70916,client_4e07408562,0.928272,STALE_CONTENT,REFRESH,104,9900.0,7461,6.9
4,5,content_2c2606c5d176,client_19581e27de,0.927673,STALE_CONTENT,REFRESH,104,590.0,104248,4.2
5,6,content_979a999506bd,client_19581e27de,0.927290,STALE_CONTENT,REFRESH,104,880.0,17216,4.0
6,7,content_90e4f1f70ab4,client_19581e27de,0.927158,STALE_CONTENT,REFRESH,104,720.0,22977,4.9
7,8,content_11fcfd65d94c,client_19581e27de,0.925515,STALE_CONTENT,REFRESH,104,480.0,44335,6.2
8,9,content_2725d2bcfac1,client_4e07408562,0.925425,STALE_CONTENT,REFRESH,104,6600.0,5915,9.1
9,10,content_d07ea098353c,client_19581e27de,0.920360,STALE_CONTENT,REFRESH,104,390.0,13698,9.4


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

path = "/content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(path))

if os.path.exists(path):
    print("✅ Dataset found!")
else:
    print("❌ Dataset still missing")
    print(os.listdir("/content/Flyrank-A.I/data/raw"))

Dataset exists: True
✅ Dataset found!


In [ ]:
!git clone https://github.com/PARIJAAT-13/Flyrank-A.I.git /content/Flyrank-A.I

fatal: destination path '/content/Flyrank-A.I' already exists and is not an empty directory.


In [ ]:
import pandas as pd
from pathlib import Path

# Load dataset
data_path = "/content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Dataset loaded: {len(df):,} rows")

# Convert required signals to numeric
numeric_cols = [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create normalized signal scores
df["staleness_score"] = df["days_since_last_update"].rank(pct=True)
df["volume_score"] = df["search_volume"].rank(pct=True)
df["impression_score"] = df["impressions_last_30d"].rank(pct=True)

# Pages ranking between positions 4–20 are actionable opportunities
df["position_score"] = (
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 20)
).astype(float)

# Baseline action score
df["baseline_score"] = (
    0.40 * df["staleness_score"]
    + 0.30 * df["volume_score"]
    + 0.20 * df["impression_score"]
    + 0.10 * df["position_score"]
)

# Reason codes
df["reason_code"] = "LOW_PRIORITY"

df.loc[
    df["staleness_score"] >= 0.75,
    "reason_code"
] = "STALE_CONTENT"

df.loc[
    (df["staleness_score"] < 0.75) &
    (df["volume_score"] >= 0.75) &
    (df["position_score"] == 1),
    "reason_code"
] = "HIGH_OPPORTUNITY"

# Action labels
df["action"] = "NO_ACTION"

df.loc[
    df["reason_code"] == "STALE_CONTENT",
    "action"
] = "REFRESH"

df.loc[
    df["reason_code"] == "HIGH_OPPORTUNITY",
    "action"
] = "REVIEW"

# Rank by score
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Required output columns
output_cols = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

queue = df[output_cols].copy()

# Save required CSV
output_path = Path(
    "/content/Flyrank-A.I/work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("✅ Ranked queue created")
print(f"Rows ranked: {len(queue):,}")
print(f"CSV saved to: {output_path}")

# Show Top 20
display(queue.head(20))

Dataset loaded: 30,000 rows
✅ Ranked queue created
Rows ranked: 30,000
CSV saved to: /content/Flyrank-A.I/work/outputs/baseline_action_score.csv


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume,impressions_last_30d,avg_position
0,1,content_5fe46e04994d,client_4e07408562,0.933433,STALE_CONTENT,REFRESH,104,1900.0,120791,4.2
1,2,content_e6955a2c59dc,client_4e07408562,0.930112,STALE_CONTENT,REFRESH,104,3600.0,11101,7.1
2,3,content_b242bb46cb5e,client_3fdba35f04,0.929402,STALE_CONTENT,REFRESH,104,1600.0,15159,5.5
3,4,content_2e0b3dc70916,client_4e07408562,0.928272,STALE_CONTENT,REFRESH,104,9900.0,7461,6.9
4,5,content_2c2606c5d176,client_19581e27de,0.927673,STALE_CONTENT,REFRESH,104,590.0,104248,4.2
5,6,content_979a999506bd,client_19581e27de,0.927290,STALE_CONTENT,REFRESH,104,880.0,17216,4.0
6,7,content_90e4f1f70ab4,client_19581e27de,0.927158,STALE_CONTENT,REFRESH,104,720.0,22977,4.9
7,8,content_11fcfd65d94c,client_19581e27de,0.925515,STALE_CONTENT,REFRESH,104,480.0,44335,6.2
8,9,content_2725d2bcfac1,client_4e07408562,0.925425,STALE_CONTENT,REFRESH,104,6600.0,5915,9.1
9,10,content_d07ea098353c,client_19581e27de,0.920360,STALE_CONTENT,REFRESH,104,390.0,13698,9.4


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Section 3 — Top-20 Review

top20 = queue.head(20).copy()

def explain_reason(row):
    if row["reason_code"] == "STALE_CONTENT":
        return (
            f"High staleness signal: content has not been updated for "
            f"{row['days_since_last_update']} days."
        )
    elif row["reason_code"] == "HIGH_OPPORTUNITY":
        return (
            f"High opportunity signal: search volume is "
            f"{row['search_volume']:.0f} and the position is "
            f"{row['avg_position']:.1f}."
        )
    else:
        return "Signals do not indicate an immediate action."

top20["why_it_is_here"] = top20.apply(explain_reason, axis=1)

top20["confidence_note"] = (
    "Moderate confidence: recommendation is based on observed "
    "decision-time signals."
)

top20["what_would_make_it_wrong"] = (
    "It could be wrong if the observed signals are incomplete, "
    "stale, or do not represent the true content opportunity."
)

review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_score",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols]

display(top20_review)

,rank,content_id,action,reason_code,baseline_score,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,REFRESH,STALE_CONTENT,0.933433,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
1,2,content_e6955a2c59dc,REFRESH,STALE_CONTENT,0.930112,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
2,3,content_b242bb46cb5e,REFRESH,STALE_CONTENT,0.929402,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
3,4,content_2e0b3dc70916,REFRESH,STALE_CONTENT,0.928272,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
4,5,content_2c2606c5d176,REFRESH,STALE_CONTENT,0.927673,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
5,6,content_979a999506bd,REFRESH,STALE_CONTENT,0.927290,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
6,7,content_90e4f1f70ab4,REFRESH,STALE_CONTENT,0.927158,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
7,8,content_11fcfd65d94c,REFRESH,STALE_CONTENT,0.925515,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
8,9,content_2725d2bcfac1,REFRESH,STALE_CONTENT,0.925425,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...
9,10,content_d07ea098353c,REFRESH,STALE_CONTENT,0.920360,High staleness signal: content has not been up...,Moderate confidence: recommendation is based o...,It could be wrong if the observed signals are ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks:
# A pick is considered potentially weak only when staleness is high
# AND both opportunity signals are relatively low.

top20 = queue.head(20).copy()

volume_threshold = top20["search_volume"].quantile(0.25)
impression_threshold = top20["impressions_last_30d"].quantile(0.25)

weak_picks = top20[
    (top20["days_since_last_update"] >= 90) &
    (top20["search_volume"] <= volume_threshold) &
    (top20["impressions_last_30d"] <= impression_threshold)
].copy()

print("=== WEAK PICKS ===")
print(f"Potential weak picks in Top-20: {len(weak_picks)}")
display(weak_picks)

=== WEAK PICKS ===
Potential weak picks in Top-20: 0


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume,impressions_last_30d,avg_position


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.